In [3]:
from pathlib import Path

input_dir = Path("/kaggle/input/")

for path in input_dir.iterdir():
    print(path)

/kaggle/input/datasets


In [4]:
data_root = next(Path("/kaggle/input").iterdir())

for item in data_root.iterdir():
    print(item)

/kaggle/input/datasets/dhirajbantawarai


In [5]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("attributes"):
    print(path)

/kaggle/input/datasets/dhirajbantawarai/camels-gb-v2-flood-data/attributes


In [6]:
attributes = next(Path("/kaggle/input").rglob("attributes"))
data_dir = attributes.parent

print("Data directory:", data_dir)
print("Attributes exists:", (data_dir / "attributes").exists())
print("Timeseries exists:", (data_dir / "timeseries").exists())

Data directory: /kaggle/input/datasets/dhirajbantawarai/camels-gb-v2-flood-data
Attributes exists: True
Timeseries exists: True


In [2]:
!pip install -q neuralhydrology==1.13.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.6/194.6 kB 4.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 44.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.1 MB/s eta 0:00:00


In [3]:
import torch
import importlib.metadata

print("NeuralHydrology:", importlib.metadata.version("neuralhydrology"))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

NeuralHydrology: 1.13.0
CUDA available: True
GPU: Tesla T4


In [4]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("final_666_haduk.yml"):
    print(path)

/kaggle/input/datasets/dhirajbantawarai/flood-zip/configs/final_666_haduk.yml


In [5]:
from pathlib import Path
import shutil

source = Path("/kaggle/input/datasets/dhirajbantawarai/flood-zip")
project = Path("/kaggle/working/floodwarningsys")

shutil.copytree(source, project, dirs_exist_ok=True)

print("Project copied to:", project)
print("Config exists:", (project / "configs/final_666_haduk.yml").exists())
print("Basin list exists:", (project / "basin_lists/final_basins.txt").exists())

Project copied to: /kaggle/working/floodwarningsys
Config exists: True
Basin list exists: True


In [6]:
from pathlib import Path
import os

project = Path("/kaggle/working/floodwarningsys")

data_source = Path(
    "/kaggle/input/datasets/dhirajbantawarai/camels-gb-v2-flood-data"
)

data_link = project / "data"

if not data_link.exists():
    os.symlink(data_source, data_link)

print("Data connected:", data_link.exists())
print("Attributes:", (data_link / "attributes").exists())
print("Timeseries:", (data_link / "timeseries").exists())

Data connected: True
Attributes: True
Timeseries: True


In [8]:
from pathlib import Path

file = Path("/kaggle/working/floodwarningsys/install_custom_nh.py")

text = file.read_text()

text = text.replace(
    r'custom_dir = r"C:\P\floodwarningsys\custom_files_new"',
    'custom_dir = os.path.join(os.path.dirname(__file__), "custom_files_new")'
)

file.write_text(text)

print("Kaggle installer updated")

Kaggle installer updated


In [9]:
%cd /kaggle/working/floodwarningsys
!python install_custom_nh.py

/kaggle/working/floodwarningsys
NeuralHydrology datasetzoo found at:
/usr/local/lib/python3.12/dist-packages/neuralhydrology/datasetzoo

Copying custom files...

Custom files successfully registered in NeuralHydrology!


In [10]:
from pathlib import Path
from neuralhydrology.datasetzoo.camelsgbv2 import CamelsGBV2

project = Path("/kaggle/working/floodwarningsys")

basins = (
    project / "basin_lists/final_basins.txt"
).read_text().splitlines()

print("Custom CamelsGBV2: OK")
print("Number of basins:", len(basins))
print("Config exists:", (project / "configs/final_666_haduk.yml").exists())
print("Data exists:", (project / "data").exists())
print("Attributes:", (project / "data/attributes").exists())
print("Timeseries:", (project / "data/timeseries").exists())

Custom CamelsGBV2: OK
Number of basins: 666
Config exists: True
Data exists: True
Attributes: True
Timeseries: True


In [11]:
import yaml

with open("configs/final_666_haduk.yml") as f:
    cfg = yaml.safe_load(f)

print("Experiment:", cfg["experiment_name"])
print("Dataset:", cfg["dataset"])
print("Data:", cfg["data_dir"])
print("Basins:", cfg["train_basin_file"])
print("Inputs:", cfg["dynamic_inputs"])
print("Model:", cfg["model"])
print("Epochs:", cfg["epochs"])
print("Batch size:", cfg["batch_size"])
print("Workers:", cfg["num_workers"])
print("Device:", cfg["device"])

Experiment: final_666_haduk.yml
Dataset: camels_gbv2
Data: data
Basins: basin_lists/final_basins.txt
Inputs: [['precipitation_haduk'], ['temperature_haduk'], ['pet_hydrope']]
Model: cudalstm
Epochs: 2
Batch size: 32
Workers: 0
Device: cuda:0


In [12]:
from pathlib import Path
from neuralhydrology.datasetzoo.camelsgbv2 import load_camels_gb_v2_timeseries

basins = Path("basin_lists/final_basins.txt").read_text().splitlines()

for basin in basins[:3]:
    df = load_camels_gb_v2_timeseries(
        Path("data"),
        basin
    )

    print(
        basin,
        "| rows:", len(df),
        "| discharge:", df["discharge_vol"].notna().sum(),
        "| rainfall:", df["precipitation_haduk"].notna().sum()
    )

10002 | rows: 18993 | discharge: 18870 | rainfall: 18993
10003 | rows: 18993 | discharge: 14381 | rainfall: 18993
1001 | rows: 18993 | discharge: 9823 | rainfall: 18993


In [13]:
%cd /kaggle/working/floodwarningsys
!nh-run train --config-file configs/final_666_haduk.yml



/kaggle/working/floodwarningsys
2026-08-18 01:03:30,761: Logging to /kaggle/working/floodwarningsys/runs/final_666_haduk.yml_1808_010330/output.log initialized.
2026-08-18 01:03:30,761: ### Folder structure created at /kaggle/working/floodwarningsys/runs/final_666_haduk.yml_1808_010330
2026-08-18 01:03:30,762: ### Run configurations for final_666_haduk.yml
2026-08-18 01:03:30,762: experiment_name: final_666_haduk.yml
2026-08-18 01:03:30,762: train_basin_file: basin_lists/final_basins.txt
2026-08-18 01:03:30,762: validation_basin_file: basin_lists/final_basins.txt
2026-08-18 01:03:30,762: test_basin_file: basin_lists/final_basins.txt
2026-08-18 01:03:30,762: train_start_date: 1970-10-01 00:00:00
2026-08-18 01:03:30,762: train_end_date: 2002-09-30 00:00:00
2026-08-18 01:03:30,762: validation_start_date: 2002-10-01 00:00:00
2026-08-18 01:03:30,762: validation_end_date: 2012-09-30 00:00:00
2026-08-18 01:03:30,762: test_start_date: 2012-10-01 00:00:00
2026-08-18 01:03:30,762: test_end_date:

In [2]:
from pathlib import Path

print("INPUTS:")
for p in Path("/kaggle/input").iterdir():
    print(p)

print("\nSAVED MODEL CHECK:")
models = list(Path("/kaggle").rglob("model_epoch002.pt"))

if models:
    for m in models:
        print(m)
else:
    print("No saved model checkpoint found.")

INPUTS:
/kaggle/input/datasets

SAVED MODEL CHECK:
No saved model checkpoint found.


In [3]:
!pip install -q neuralhydrology==1.13.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.6/194.6 kB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 49.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.2 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import shutil
import os

code = Path("/kaggle/input/datasets/dhirajbantawarai/flood-zip")
project = Path("/kaggle/working/floodwarningsys")

shutil.copytree(code, project, dirs_exist_ok=True)

data = Path(
    "/kaggle/input/datasets/dhirajbantawarai/camels-gb-v2-flood-data"
)

os.symlink(data, project / "data")

print("Project:", project.exists())
print("Data:", (project / "data").exists())

Project: True
Data: True


In [5]:
%cd /kaggle/working/floodwarningsys

!python install_custom_nh.py

/kaggle/working/floodwarningsys
NeuralHydrology datasetzoo found at:
/usr/local/lib/python3.12/dist-packages/neuralhydrology/datasetzoo

Copying custom files...
Traceback (most recent call last):
  File "/kaggle/working/floodwarningsys/install_custom_nh.py", line 37, in <module>
    shutil.copy(
  File "/usr/lib/python3.12/shutil.py", line 435, in copy
    copyfile(src, dst, follow_symlinks=follow_symlinks)
  File "/usr/lib/python3.12/shutil.py", line 260, in copyfile
    with open(src, 'rb') as fsrc:
         ^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'C:\\P\\floodwarningsys\\custom_files_new/camelsgbv2h.py'


In [6]:
from pathlib import Path

file = Path("/kaggle/working/floodwarningsys/install_custom_nh.py")

text = file.read_text()

text = text.replace(
    r'custom_dir = r"C:\P\floodwarningsys\custom_files_new"',
    'custom_dir = os.path.join(os.path.dirname(__file__), "custom_files_new")'
)

file.write_text(text)

print("Installer fixed for Kaggle")

Installer fixed for Kaggle


In [7]:
%cd /kaggle/working/floodwarningsys

!python install_custom_nh.py

/kaggle/working/floodwarningsys
NeuralHydrology datasetzoo found at:
/usr/local/lib/python3.12/dist-packages/neuralhydrology/datasetzoo

Copying custom files...

Custom files successfully registered in NeuralHydrology!


In [8]:
from pathlib import Path

config = Path(
    "/kaggle/working/floodwarningsys/configs/final_666_haduk.yml"
)

text = config.read_text()

text = text.replace(
    "experiment_name: final_666_haduk.yml",
    "experiment_name: final_666_haduk"
)

config.write_text(text)

print("Config updated")

Config updated


In [9]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA: True
GPU: Tesla T4


In [21]:
from pathlib import Path

config = Path(
    "/kaggle/working/floodwarningsys/configs/final_666_haduk.yml"
)

text = config.read_text()

# Change epochs
text = text.replace(
    "epochs: 10",
    "epochs: 6"
)

# Update learning-rate schedule for 6 epochs
start = text.index("learning_rate:")
end = text.index("\nbatch_size:", start)

new_lr = """learning_rate:
  0: 0.001
  3: 0.0005
  5: 0.0001
"""

text = text[:start] + new_lr + text[end:]

config.write_text(text)

print("Config changed to 6 epochs")

Config changed to 6 epochs


In [22]:
import yaml

with open(
    "/kaggle/working/floodwarningsys/configs/final_666_haduk.yml"
) as f:
    cfg = yaml.safe_load(f)

print("Epochs:", cfg["epochs"])
print("Learning rate:", cfg["learning_rate"])

Epochs: 6
Learning rate: {0: 0.001, 3: 0.0005, 5: 0.0001}


In [ ]:
%cd /kaggle/working/floodwarningsys

!nh-run train --config-file configs/final_666_haduk.yml

/kaggle/working/floodwarningsys
2026-08-18 12:52:12,936: Logging to /kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/output.log initialized.
2026-08-18 12:52:12,936: ### Folder structure created at /kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212
2026-08-18 12:52:12,936: ### Run configurations for final_666_haduk
2026-08-18 12:52:12,937: experiment_name: final_666_haduk
2026-08-18 12:52:12,937: train_basin_file: basin_lists/final_basins.txt
2026-08-18 12:52:12,937: validation_basin_file: basin_lists/final_basins.txt
2026-08-18 12:52:12,937: test_basin_file: basin_lists/final_basins.txt
2026-08-18 12:52:12,937: train_start_date: 1970-10-01 00:00:00
2026-08-18 12:52:12,937: train_end_date: 2002-09-30 00:00:00
2026-08-18 12:52:12,937: validation_start_date: 2002-10-01 00:00:00
2026-08-18 12:52:12,937: validation_end_date: 2012-09-30 00:00:00
2026-08-18 12:52:12,937: test_start_date: 2012-10-01 00:00:00
2026-08-18 12:52:12,937: test_end_date: 2022-09-30 00:0

In [24]:
from pathlib import Path

models = sorted(
    Path("/kaggle/working/floodwarningsys/runs").rglob("model_epoch*.pt")
)

print("Checkpoints found:", len(models))

for model in models:
    print(model)

Checkpoints found: 3
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/model_epoch001.pt
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/model_epoch002.pt
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/model_epoch003.pt


In [ ]:
from pathlib import Path
from neuralhydrology.nh_run import continue_run

run_dir = Path(
    "/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212"
)

continue_run(run_dir=run_dir)

2026-08-18 15:23:51,158: Logging to /kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/continue_training_from_epoch003/output.log initialized.
2026-08-18 15:23:51,160: ### Folder structure created at /kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/continue_training_from_epoch003
2026-08-18 15:23:51,161: ### Continue training of run stored in /kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212
2026-08-18 15:23:51,162: ### Run configurations for final_666_haduk
2026-08-18 15:23:51,162: base_run_dir: /kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212
2026-08-18 15:23:51,163: batch_size: 32
2026-08-18 15:23:51,165: clip_gradient_norm: 1
2026-08-18 15:23:51,166: clip_targets_to_zero: ['discharge_vol']
2026-08-18 15:23:51,167: commit_hash: None
2026-08-18 15:23:51,167: data_dir: data
2026-08-18 15:23:51,168: dataset: camels_gbv2
2026-08-18 15:23:51,169: device: cuda:0
2026-08-18 15:23:51,169: dynamic_inputs: [['precipitation_haduk'], 

In [26]:
from pathlib import Path

run = Path(
    "/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212"
)

for epoch in range(1, 7):
    model = run / f"model_epoch{epoch:03d}.pt"
    print(f"Epoch {epoch}:", model.exists())

Epoch 1: True
Epoch 2: True
Epoch 3: True
Epoch 4: False
Epoch 5: False
Epoch 6: False


In [27]:
from pathlib import Path

run = Path(
    "/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212"
)

models = sorted(run.rglob("model_epoch*.pt"))

print("Checkpoints found:", len(models))

for model in models:
    print(model)

Checkpoints found: 6
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/continue_training_from_epoch003/model_epoch004.pt
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/continue_training_from_epoch003/model_epoch005.pt
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/continue_training_from_epoch003/model_epoch006.pt
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/model_epoch001.pt
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/model_epoch002.pt
/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212/model_epoch003.pt


In [28]:
from pathlib import Path
import shutil

run = Path(
    "/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212"
)

backup = "/kaggle/working/final_666_haduk_complete_backup"

shutil.make_archive(
    backup,
    "zip",
    run
)

print("Backup created:")
print(backup + ".zip")

Backup created:
/kaggle/working/final_666_haduk_complete_backup.zip


In [29]:
from pathlib import Path

backup = Path(
    "/kaggle/working/final_666_haduk_complete_backup.zip"
)

print("Backup exists:", backup.exists())
print("Size MB:", round(backup.stat().st_size / 1024 / 1024, 2))

Backup exists: True
Size MB: 17.7


In [30]:
from pathlib import Path
import pandas as pd

run = Path(
    "/kaggle/working/floodwarningsys/runs/final_666_haduk_1808_125212"
)

results = []

for file in run.rglob("validation_metrics.csv"):

    epoch = int(
        file.parent.name.replace("model_epoch", "")
    )

    df = pd.read_csv(file)

    results.append({
        "Epoch": epoch,
        "Median NSE": df["NSE"].median(),
        "Median KGE": df["KGE"].median()
    })

summary = (
    pd.DataFrame(results)
    .sort_values("Epoch")
)

print(summary.to_string(index=False))

 Epoch  Median NSE  Median KGE
     1    0.698643    0.703447
     2    0.667645    0.734081
     3    0.798999    0.763531
     4    0.786903    0.754797
     5    0.751954    0.795721
     6    0.871937    0.885407


In [31]:
%cd /kaggle/working/floodwarningsys

!nh-run evaluate \
--run-dir runs/final_666_haduk_1808_125212/continue_training_from_epoch003 \
--epoch 6

/kaggle/working/floodwarningsys
2026-08-18 17:38:00,925: Logging to runs/final_666_haduk_1808_125212/continue_training_from_epoch003/output.log initialized.
2026-08-18 17:38:01,436: Using the model weights from runs/final_666_haduk_1808_125212/continue_training_from_epoch003/model_epoch006.pt
# Evaluation:   0%|                                     | 0/666 [00:00<?, ?it/s]2026-08-18 17:38:01,499: CAMELS_GB_V2 attribute files loaded (7): ['camels_gb_v2_topographic_attributes.csv', 'camels_gb_v2_climatic_attributes.csv', 'camels_gb_v2_landcover_attributes.csv', 'camels_gb_v2_hydrogeology_attributes.csv', 'camels_gb_v2_hydrologic_attributes.csv', 'camels_gb_v2_humaninfluence_attributes.csv', 'camels_gb_v2_soil_attributes.csv']
2026-08-18 17:38:01,500: CAMELS_GB_V2 attribute files skipped (no gauge_id index, 1): ['camels_gb_v2_groundwaterwell_attributes.csv']
# Evaluation:   0%|                             | 1/666 [00:03<35:31,  3.21s/it]2026-08-18 17:38:04,688: CAMELS_GB_V2 attribute files

In [32]:
from IPython.display import FileLink

FileLink(
    "/kaggle/working/final_666_haduk_complete_backup.zip"
)

/kaggle/working/final_666_haduk_complete_backup.zip

In [33]:
%cd /kaggle/working

from IPython.display import FileLink

FileLink("final_666_haduk_complete_backup.zip")

/kaggle/working


/kaggle/working/final_666_haduk_complete_backup.zip

In [1]:
from pathlib import Path

checkpoint = Path(
    "/kaggle/working/floodwarningsys/runs/"
    "final_666_haduk_1808_125212/"
    "continue_training_from_epoch003/"
    "model_epoch006.pt"
)

print("Epoch 6 checkpoint exists:", checkpoint.exists())


Epoch 6 checkpoint exists: False


In [2]:
from pathlib import Path

models = list(
    Path("/kaggle/input").rglob("model_epoch006.pt")
)

print("Epoch 6 checkpoints found:", len(models))

for model in models:
    print(model)

Epoch 6 checkpoints found: 1
/kaggle/input/datasets/dhirajbantawarai/final-666-haduk-backup/continue_training_from_epoch003/model_epoch006.pt


In [3]:
from pathlib import Path
import shutil
import os

project = Path("/kaggle/working/floodwarningsys")

# Restore project code
code_source = Path(
    "/kaggle/input/datasets/dhirajbantawarai/flood-zip"
)

shutil.copytree(
    code_source,
    project,
    dirs_exist_ok=True
)

# Connect CAMELS data
data_source = Path(
    "/kaggle/input/datasets/dhirajbantawarai/camels-gb-v2-flood-data"
)

data_link = project / "data"

if not data_link.exists():
    os.symlink(data_source, data_link)

# Restore trained run
backup_source = Path(
    "/kaggle/input/datasets/dhirajbantawarai/final-666-haduk-backup"
)

run_target = (
    project
    / "runs"
    / "final_666_haduk_1808_125212"
)

shutil.copytree(
    backup_source,
    run_target,
    dirs_exist_ok=True
)

print("Project:", project.exists())
print("Data:", data_link.exists())

checkpoint = (
    run_target
    / "continue_training_from_epoch003"
    / "model_epoch006.pt"
)

print("Epoch 6:", checkpoint.exists())

Project: True
Data: True
Epoch 6: True


In [4]:
!pip install -q neuralhydrology==1.13.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.6/194.6 kB 4.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 80.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.5 MB/s eta 0:00:00


In [5]:
from pathlib import Path

file = Path("/kaggle/working/floodwarningsys/install_custom_nh.py")

text = file.read_text()

text = text.replace(
    r'custom_dir = r"C:\P\floodwarningsys\custom_files_new"',
    'custom_dir = os.path.join(os.path.dirname(__file__), "custom_files_new")'
)

file.write_text(text)

print("Installer fixed")

Installer fixed


In [6]:
%cd /kaggle/working/floodwarningsys

!python install_custom_nh.py

/kaggle/working/floodwarningsys
NeuralHydrology datasetzoo found at:
/usr/local/lib/python3.12/dist-packages/neuralhydrology/datasetzoo

Copying custom files...

Custom files successfully registered in NeuralHydrology!


In [7]:
import torch
from pathlib import Path
from neuralhydrology.datasetzoo.camelsgbv2 import CamelsGBV2

checkpoint = Path(
    "runs/final_666_haduk_1808_125212/"
    "continue_training_from_epoch003/"
    "model_epoch006.pt"
)

print("Custom CamelsGBV2: OK")
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Epoch 6:", checkpoint.exists())

Custom CamelsGBV2: OK
CUDA: True
GPU: Tesla T4
Epoch 6: True


In [8]:
%cd /kaggle/working/floodwarningsys

!nh-run evaluate \
--run-dir runs/final_666_haduk_1808_125212/continue_training_from_epoch003 \
--epoch 6

/kaggle/working/floodwarningsys
2026-08-19 11:26:35,824: Logging to runs/final_666_haduk_1808_125212/continue_training_from_epoch003/output.log initialized.
2026-08-19 11:26:36,886: Using the model weights from runs/final_666_haduk_1808_125212/continue_training_from_epoch003/model_epoch006.pt
# Evaluation:   0%|                                     | 0/666 [00:00<?, ?it/s]2026-08-19 11:26:37,062: CAMELS_GB_V2 attribute files loaded (7): ['camels_gb_v2_topographic_attributes.csv', 'camels_gb_v2_climatic_attributes.csv', 'camels_gb_v2_landcover_attributes.csv', 'camels_gb_v2_hydrogeology_attributes.csv', 'camels_gb_v2_hydrologic_attributes.csv', 'camels_gb_v2_humaninfluence_attributes.csv', 'camels_gb_v2_soil_attributes.csv']
2026-08-19 11:26:37,062: CAMELS_GB_V2 attribute files skipped (no gauge_id index, 1): ['camels_gb_v2_groundwaterwell_attributes.csv']
# Evaluation:   0%|                           | 1/666 [00:05<1:05:35,  5.92s/it]2026-08-19 11:26:42,865: CAMELS_GB_V2 attribute files

In [9]:
import pandas as pd
from pathlib import Path

file = Path(
    "runs/final_666_haduk_1808_125212/"
    "continue_training_from_epoch003/"
    "test/model_epoch006/test_metrics.csv"
)

df = pd.read_csv(file)

print("Rows in metrics file:", len(df))
print("Valid NSE:", df["NSE"].notna().sum())
print("Valid KGE:", df["KGE"].notna().sum())

print("\nMedian NSE:", df["NSE"].median())
print("Median KGE:", df["KGE"].median())

print("\nNSE > 0:", (df["NSE"] > 0).sum())
print("NSE > 0.5:", (df["NSE"] > 0.5).sum())
print("NSE > 0.7:", (df["NSE"] > 0.7).sum())

Rows in metrics file: 666
Valid NSE: 662
Valid KGE: 662

Median NSE: 0.8357484056977511
Median KGE: 0.7946410072011887

NSE > 0: 660
NSE > 0.5: 649
NSE > 0.7: 589


In [10]:
from pathlib import Path
import shutil

results = Path(
    "/kaggle/working/floodwarningsys/runs/"
    "final_666_haduk_1808_125212/"
    "continue_training_from_epoch003/test/model_epoch006"
)

output = "/kaggle/working/final_666_epoch6_evaluation"

shutil.make_archive(
    output,
    "zip",
    results
)

print("Created:")
print(output + ".zip")

Created:
/kaggle/working/final_666_epoch6_evaluation.zip


In [11]:
from pathlib import Path

file = Path(
    "/kaggle/working/final_666_epoch6_evaluation.zip"
)

print("Exists:", file.exists())
print("Size MB:", round(file.stat().st_size / 1024 / 1024, 2))

Exists: True
Size MB: 25.6


In [12]:
%cd /kaggle/working

from IPython.display import FileLink

FileLink("final_666_epoch6_evaluation.zip")

/kaggle/working


/kaggle/working/final_666_epoch6_evaluation.zip